In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

import sys
from pathlib import Path

sys.path.append(str(Path("../src").resolve()))   # folder that CONTAINS the .py file

import utils
from utils import clean, get_data, get_results             # for src/my_utils.py


In [2]:
df_2024 = pd.read_csv("../data/df_2024_final.csv").assign(season=2024)
df_2025 = pd.read_csv("../data/df_2025_final.csv").assign(season=2025)
df = pd.concat([df_2024, df_2025], ignore_index=True)

In [3]:
df.head()
df= df.drop(columns="Unnamed: 0")

In [ ]:

## necessary columns for each position to define separate data frames
id_cols = ['player_id', 'player', 'position', 'team', 'season', 'week',
           'date', 'opponent', 'atVs']

context_cols = ['opp_def_rank', 'team_pass_rank', 'team_rush_rank',
                'stadium', 'roof', 'surface', 'temp', 'wind', 'location',
                'gametime']

target_cols = ['fantasy_points', 'projected_points']



rushing = ['rushingAttempts', 'rushingYards', 'yardsPerRushAttempt',
           'rushingTouchdowns', 'longRushing']
receiving = ['receptions', 'receivingTargets', 'receivingYards',
             'yardsPerReception', 'receivingTouchdowns', 'longReception']
ball_security = ['fumbles', 'fumblesLost']
passing = ['completions', 'passingAttempts', 'passingYards', 'completionPct',
           'yardsPerPassAttempt', 'passingTouchdowns', 'interceptions',
           'longPassing', 'sacks', 'QBRating', 'adjQBR']
kicking = ['fieldGoalsMade', 'fieldGoalAttempts', 'fieldGoalPct',
           'longFieldGoalMade', 'fieldGoalsMadeYardsAverage',
           'fieldGoalsMade1_19', 'fieldGoalAttempts1_19',
           'fieldGoalsMade20_29', 'fieldGoalAttempts20_29',
           'fieldGoalsMade30_39', 'fieldGoalAttempts30_39',
           'fieldGoalsMade40_49', 'fieldGoalAttempts40_49',
           'fieldGoalsMade50', 'fieldGoalAttempts50',
           'extraPointsMade', 'extraPointAttempts', 'totalKickingPoints']

base = id_cols + context_cols + target_cols

position_cols = {
    'QB': base + passing + rushing + ball_security,
    'WR': base + receiving + rushing + ball_security,
    'TE': base + receiving + rushing + ball_security,
    'K':  base + kicking,
    "RB": base + rushing + receiving + ball_security
}
## generate each postional dataframe based on what scoring metrics contribute to their respective fantasy score
pos_dfs = {pos: df.loc[df['position'] == pos, cols].copy()
           for pos, cols in position_cols.items()}

qb_df, wr_df, te_df, k_df, rb_df = (pos_dfs[p] for p in ['QB', 'WR', 'TE', 'K', 'RB'])

In [6]:
## combined data frames based on cluster
qb_te = pd.concat([qb_df, te_df])
#qb_te.head()


In [7]:
wr_k = pd.concat([wr_df, k_df])

In [8]:
# now we need to sort each data frame in order
rb_df = rb_df.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)
qb_te = qb_te.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)
wr_k = wr_k.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)


In [9]:
## need to shift by -1 to create target of next weeks fantasy score
g_1 = rb_df.groupby(['player_id', "season"])
rb_df["target_fantasy_points"] = g_1["fantasy_points"].shift(-1)
rb_df['next_week_opponent'] = g_1['opponent'].shift(-1)
rb_df['next_week_atVs'] = g_1['atVs'].shift(-1)
rb_df['next_week_projected'] = g_1['projected_points'].shift(-1) 

g_2 = qb_te.groupby(['player_id', "season"])
qb_te["target_fantasy_points"] = g_2["fantasy_points"].shift(-1)
qb_te['next_week_opponent'] = g_2['opponent'].shift(-1)
qb_te['next_week_atVs'] = g_2['atVs'].shift(-1)
qb_te['next_week_projected'] = g_2['projected_points'].shift(-1) 

g_3 = wr_k.groupby(['player_id', "season"])
wr_k["target_fantasy_points"] = g_3["fantasy_points"].shift(-1)
wr_k["next_week_opponent"] = g_3["opponent"].shift(-1)
wr_k["next_week_atVs"] = g_3["atVs"].shift(-1)
wr_k["next_week_projected"] = g_3["projected_points"].shift(-1)
for col in context_cols:
    rb_df[f"next_week_{col}"] = g_1[col].shift(-1)
    qb_te[f"next_week_{col}"] = g_2[col].shift(-1)
    wr_k[f"next_week_{col}"] = g_3[col].shift(-1)


In [ ]:
## encode categorical columns


shared_cat = ['position', 'next_week_opponent', 'next_week_atVs', 'next_week_surface',
                    'next_week_roof', 'next_week_location', 'next_week_stadium', 'next_week_gametime', "team"]
target = "target_fantasy_points"

rb_df[shared_cat] = rb_df[shared_cat].astype("category")
qb_te[shared_cat] = qb_te[shared_cat].astype("category")
wr_k[shared_cat] = wr_k[shared_cat].astype("category")

In [ ]:
# generate training splits for each group
train_data_1 = rb_df[(rb_df["season"] == 2024) | ((rb_df["season"] == 2025) & (rb_df["week"] < 7))]
train_data_2 = qb_te[(qb_te["season"] == 2024) | ((qb_te["season"] == 2025) & (qb_te["week"] < 7))]
train_data_3 = wr_k[(wr_k["season"] == 2024) | ((wr_k["season"] == 2025) & (wr_k["week"] < 7))]

val_1 = rb_df[(rb_df["season"] == 2025) & (rb_df["week"].between(7,9))]
val_2 = qb_te[(qb_te["season"] == 2025) & (qb_te["week"].between(7,9))]
val_3 = wr_k[(wr_k["season"] == 2025) & (wr_k["week"].between(7,9))]

test_1 =  rb_df[(rb_df["season"] == 2025) & (rb_df["week"] >= 10)]
test_2 =  qb_te[(qb_te["season"] == 2025) & (qb_te["week"] >= 10)]
test_3 = wr_k[(wr_k["season"] ==2025) & (wr_k["week"] >= 10)]


In [ ]:
## common columns across all the data frames
common = list(set(rb_df.columns) & set(qb_te.columns) & set(wr_k.columns))
irrelevant_columns = [
    'player_id', 'player', 'opponent', "date",
    'eventId', 'game_id', 'gameday', 'gametime', 'stadium', 'result',
    'score', 'fantasy_points', 'target_next_week_fantasy', 'espn',
    '_set', 'season',
    'med_proj_pts',
    'projected_points',
    'atVs', 'roof', 'surface', 'temp', 'wind', 'location',
    'opp_def_rank', 'team_pass_rank', 'team_rush_rank', 'next_week_projected' 
]

# want to get all irrelevant columns that dont contribute to score
irrelevant = list(set(common) & set(irrelevant_columns))

In [15]:


## Running back only
X1_train, y1_train = get_data(train_data_1, irrelevant, target)
X2_train, y2_train = get_data(train_data_2, irrelevant, target)
X3_train, y3_train = get_data(train_data_3, irrelevant, target)

##quarter back/ tight end
X1_val, y1_val = get_data(val_1, irrelevant, target)
X2_val, y2_val = get_data(val_2, irrelevant, target)
X3_val, y3_val = get_data(val_3, irrelevant, target)
## Wide reciever/kicker
X1_test, y1_test = get_data(test_1, irrelevant, target)
X2_test, y2_test = get_data(test_2, irrelevant, target)
X3_test, y3_test = get_data(test_3, irrelevant, target)

In [16]:
model1 = xgb.XGBRegressor(
    tree_method = "hist",
    n_estimators=10000, #Higher is better
    learning_rate=0.005, #lower is better
    max_depth=5, #higher is better
    min_child_weight=3, #mid:1-10
    subsample=0.7, #mid:0-1
    colsample_bytree=0.7, #mid:0-1
    random_state=12, #seed
    enable_categorical=True,
    early_stopping_rounds=50      # stops when validation stops improving (old run overfit after ~50 trees)
)



In [17]:
model1.fit(
    X1_train, y1_train,
    eval_set=[(X1_val, y1_val)],
    verbose=50 #how often to print validation score 
)

[0]	validation_0-rmse:8.21894
[50]	validation_0-rmse:6.80494
[100]	validation_0-rmse:5.68044
[150]	validation_0-rmse:4.84876
[200]	validation_0-rmse:4.13559
[250]	validation_0-rmse:3.58584
[300]	validation_0-rmse:3.12895
[350]	validation_0-rmse:2.73651
[400]	validation_0-rmse:2.43103
[450]	validation_0-rmse:2.20260
[500]	validation_0-rmse:2.03428
[550]	validation_0-rmse:1.86463
[600]	validation_0-rmse:1.71618
[650]	validation_0-rmse:1.61200
[700]	validation_0-rmse:1.53755
[750]	validation_0-rmse:1.46125
[800]	validation_0-rmse:1.40604
[850]	validation_0-rmse:1.36313
[900]	validation_0-rmse:1.32204
[950]	validation_0-rmse:1.28996
[1000]	validation_0-rmse:1.25749
[1050]	validation_0-rmse:1.22881
[1100]	validation_0-rmse:1.21196
[1150]	validation_0-rmse:1.19441
[1200]	validation_0-rmse:1.18079
[1250]	validation_0-rmse:1.16995
[1300]	validation_0-rmse:1.16146
[1350]	validation_0-rmse:1.15225
[1400]	validation_0-rmse:1.14355
[1450]	validation_0-rmse:1.13523
[1500]	validation_0-rmse:1.12930


,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.7
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [18]:
model2 = xgb.XGBRegressor(
    tree_method = "hist",
    n_estimators=10000, #Higher is better
    learning_rate=0.005, #lower is better
    max_depth=5, #higher is better
    min_child_weight=3, #mid:1-10
    subsample=0.7, #mid:0-1
    colsample_bytree=0.7, #mid:0-1
    random_state=12, #seed
    enable_categorical=True,
    early_stopping_rounds=50      # stops when validation stops improving (old run overfit after ~50 trees)
)
model2.fit(X2_train, y2_train,
        eval_set=[(X2_val, y2_val)],
        verbose=50)

[0]	validation_0-rmse:8.31979
[50]	validation_0-rmse:6.74537
[100]	validation_0-rmse:5.53798
[150]	validation_0-rmse:4.58918
[200]	validation_0-rmse:3.80799
[250]	validation_0-rmse:3.17594
[300]	validation_0-rmse:2.71307
[350]	validation_0-rmse:2.34636
[400]	validation_0-rmse:2.02607
[450]	validation_0-rmse:1.82961
[500]	validation_0-rmse:1.64833
[550]	validation_0-rmse:1.47746
[600]	validation_0-rmse:1.33268
[650]	validation_0-rmse:1.20589
[700]	validation_0-rmse:1.11748
[750]	validation_0-rmse:1.04719
[800]	validation_0-rmse:0.98299
[850]	validation_0-rmse:0.94144
[900]	validation_0-rmse:0.90442
[950]	validation_0-rmse:0.87285
[1000]	validation_0-rmse:0.84792
[1050]	validation_0-rmse:0.82303
[1100]	validation_0-rmse:0.79822
[1150]	validation_0-rmse:0.77992
[1200]	validation_0-rmse:0.76475
[1250]	validation_0-rmse:0.75079
[1300]	validation_0-rmse:0.73688
[1350]	validation_0-rmse:0.72592
[1400]	validation_0-rmse:0.71686
[1450]	validation_0-rmse:0.70816
[1500]	validation_0-rmse:0.70157


,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.7
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [19]:
model3 = xgb.XGBRegressor(
    tree_method = "hist",
    n_estimators=10000, #Higher is better
    learning_rate=0.005, #lower is better
    max_depth=5, #higher is better
    min_child_weight=3, #mid:1-10
    subsample=0.7, #mid:0-1
    colsample_bytree=0.7, #mid:0-1
    random_state=12, #seed
    enable_categorical=True,
    early_stopping_rounds=50      # stops when validation stops improving (old run overfit after ~50 trees)
)
model3.fit(X3_train, y3_train,
        eval_set=[(X3_val, y3_val)],
        verbose=50)

[0]	validation_0-rmse:6.82700
[50]	validation_0-rmse:5.60711
[100]	validation_0-rmse:4.66538
[150]	validation_0-rmse:3.81240
[200]	validation_0-rmse:3.15281
[250]	validation_0-rmse:2.63434
[300]	validation_0-rmse:2.26620
[350]	validation_0-rmse:1.88186
[400]	validation_0-rmse:1.63217
[450]	validation_0-rmse:1.46462
[500]	validation_0-rmse:1.28950
[550]	validation_0-rmse:1.14077
[600]	validation_0-rmse:1.01724
[650]	validation_0-rmse:0.91997
[700]	validation_0-rmse:0.83001
[750]	validation_0-rmse:0.74967
[800]	validation_0-rmse:0.68117
[850]	validation_0-rmse:0.63453
[900]	validation_0-rmse:0.60031
[950]	validation_0-rmse:0.56683
[1000]	validation_0-rmse:0.53374
[1050]	validation_0-rmse:0.50409
[1100]	validation_0-rmse:0.48007
[1150]	validation_0-rmse:0.45998
[1200]	validation_0-rmse:0.44002
[1250]	validation_0-rmse:0.42336
[1300]	validation_0-rmse:0.41105
[1350]	validation_0-rmse:0.39741
[1400]	validation_0-rmse:0.38752
[1450]	validation_0-rmse:0.37923
[1500]	validation_0-rmse:0.37134


,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.7
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [20]:
## get our results for model 1
rb_results = get_results(rb_df, X1_test, y1_test, model1)

Model: MAE=0.425  RMSE=1.008  R2=0.985
ESPN: MAE=3.905  RMSE=5.852  R2=0.478


In [21]:
# model 2 results
qb_te_results=get_results(qb_te, X2_test, y2_test, model2)

Model: MAE=0.361  RMSE=0.721  R2=0.992
ESPN: MAE=3.716  RMSE=5.589  R2=0.513


In [22]:
wr_k_results = get_results(wr_k, X3_test, y3_test, model3)

Model: MAE=0.217  RMSE=0.454  R2=0.996
ESPN: MAE=3.968  RMSE=5.492  R2=0.378


In [23]:
all_pos = pd.concat([rb_results, qb_te_results, wr_k_results], ignore_index=True)
position_table = pd.crosstab(all_pos['position'], all_pos['winner'], margins=True)
position_table

winner,Model,espn,All
position,,,
K,232,4,236
QB,242,29,271
RB,644,131,775
TE,618,121,739
WR,1054,102,1156
All,2790,387,3177
